# LLM 성능평가

LLM 성능평가는 모델이 낸 답이 과업의 성공 조건을 얼마나 만족하는지 증거로 확인하는 과정이다. 번역·요약·질의응답은 정답의 형태가 서로 다르므로 하나의 점수만으로 모든 품질을 설명할 수 없다.

이 노트북은 로컬 CPU에서 실행할 수 있으며 모델 다운로드, GPU, Hugging Face token과 외부 LLM API key가 필요하지 않다. `평가 목적 정의 → 벤치마크 선택 → 지표 계산 → 채점 방식 선택 → 자동화 → LLM 심사 → 오류 분석` 순서로 평가 설계의 뼈대를 익힌다.


## 평가 설계의 출발점

평가 전에 먼저 성공을 정의한다. 번역은 의미 보존과 자연스러움, 요약은 핵심 정보 보존과 압축, 고객 응대는 정책 준수와 문제 해결을 함께 본다.

개발용 데이터로 프롬프트를 고친 뒤에는 별도의 고정 테스트셋으로 다시 측정한다. 표준 벤치마크에는 모델 간 비교를 맡기고, 실제 서비스 입력으로 만든 평가셋에는 배포 적합성 판단을 맡긴다. 학습 데이터 오염, 언어·도메인 차이, 프롬프트와 few-shot 수가 점수에 영향을 줄 수 있으므로 데이터셋 버전과 실행 조건도 함께 기록한다.

### 하나의 질문도 채점 기준에 따라 결과가 달라진다

질문이 `대한민국의 수도는 어디인가?`, 기준 답이 `서울`이라고 가정한다.

| 후보 답변 | Exact Match | 의미·사실 기준 | 해석 |
|---|---:|---:|---|
| `서울` | 정답 | 정답 | 기준 문자열과 의미가 모두 일치한다. |
| `대한민국의 수도는 서울이다.` | 오답 | 정답 | 표현은 다르지만 의미는 같다. |
| `부산` | 오답 | 오답 | 문자열과 사실이 모두 틀리다. |

이 사례는 정확한 라벨 비교, 문자열 겹침, 의미 평가, 사람 또는 LLM 심사가 서로 다른 역할을 맡는 이유를 보여 준다.


## LLM 벤치마크

벤치마크는 동일한 문제와 채점 규칙으로 모델의 특정 능력을 비교하는 표준 시험이다. 수능 점수가 학생의 모든 능력을 말해 주지 않듯 벤치마크 점수도 실제 서비스 품질 전체를 보증하지 않는다.

| 평가 관점 | 확인하는 질문 | 대표 대상 |
|---|---|---|
| 일반 성능 | 여러 모델에 공통인 지식·언어 이해·추론·독해 능력이 있는가? | MMLU, ARC, SQuAD |
| 도메인 특화 | 금융·법률·의료 등 특정 업무 지식과 판단이 가능한가? | 업무별 자체 평가셋, 전문 시험형 벤치마크 |
| 얼라인먼트·안전 | 지시와 정책을 따르고 편향·유해 출력을 제어하는가? | IFEval, 안전·편향 평가셋, 사람 평가 |

세 관점은 서로 대체하지 않는다. 일반 지식 점수가 높아도 특정 회사의 업무 규칙이나 안전 정책을 잘 따르는지는 별도로 확인해야 한다.


### Llama 3.1 모델 카드에서 벤치마크 읽기

Meta의 Llama 3.1 모델 카드는 base 모델과 instruction-tuned 모델의 평가표를 구분한다. 아래 범주는 모든 LLM에 적용되는 절대 분류가 아니라 해당 모델 카드의 결과를 읽기 위한 구조이다.

| 모델 카드 영역 | 대표 벤치마크 | 주로 확인하는 능력 |
|---|---|---|
| Base - General | MMLU, AGIEval, ARC-Challenge | 지식과 추론 |
| Base - Knowledge·Reading | TriviaQA, SQuAD, BoolQ, DROP | 지식 회상과 독해 |
| Instruct - Reasoning·Code·Math | GPQA, HumanEval, GSM8K, MATH | 과학·코드·수학 |
| Instruct - Tool Use·Multilingual | API-Bank, BFCL, Multilingual MMLU | 도구 호출과 다국어 |

숫자를 비교할 때는 모델 종류, 데이터셋 버전, 샘플 수, prompt, few-shot 수, CoT 사용 여부, metric이 같은지 확인한다. 모델 카드 점수는 후보를 좁히는 출발점이며 한국어 업무 문서나 실제 RAG 답변의 품질을 보증하지 않는다.

출처: [Meta Llama 3.1 8B Instruct 모델 카드](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)


## 한국어 벤치마크

한국어 벤치마크는 한국어 문맥 이해, 시험형 지식, 문서 독해처럼 서로 다른 능력을 측정한다. 이름만 외우기보다 `입력 → 모델이 내야 할 출력 → 채점 기준`을 구분해야 한다.

| 벤치마크 | 입력 | 기대 출력 | 대표 채점 |
|---|---|---|---|
| KoBEST | 문단·문장·선택지 | 참/거짓 라벨 또는 선택지 | Accuracy, F1 |
| KMMLU | 질문과 네 개 선택지 | 정답 선택지 | Accuracy |
| KorQuAD | 문서와 질문 | 문서 안의 답 구간 | Exact Match, token F1 |


### KoBEST: 문맥을 읽고 라벨이나 선택지를 판단한다

KoBEST는 한국어 자연어 이해를 평가하는 다섯 과업으로 구성된다.

| 과업 | 모델이 판단하는 내용 |
|---|---|
| BoolQ | 문단을 근거로 질문의 참·거짓을 판단한다. |
| COPA | 전제에 가장 적절한 원인 또는 결과를 선택한다. |
| WiC | 두 문맥에서 목표 단어의 의미가 같은지 판단한다. |
| HellaSwag | 문맥 다음에 이어질 자연스러운 문장을 선택한다. |
| SentiNeg | 부정 표현을 포함한 문장의 감성 극성을 분류한다. |

다음 KB-BoolQ 사례는 문단과 질문을 입력으로 받고 참·거짓 라벨을 정답으로 사용한다.

```python
{
    "paragraph": "두아 리파는 잉글랜드의 싱어송라이터이자 모델이다.",
    "question": "두아 리파는 영국인인가?",
    "label": 1,
}
```

BoolQ 설정에서 `0`은 거짓, `1`은 참이다. 모델이 자연어로 길게 답하더라도 평가는 최종 라벨이나 두 후보의 점수만 사용할 수 있다.

출처: [KoBEST 원 논문](https://aclanthology.org/2022.coling-1.325/), [KoBEST 데이터 카드](https://huggingface.co/datasets/skt/kobest_v1)


### KMMLU: 한국어 시험형 객관식 문제를 푼다

KMMLU는 인문사회부터 STEM까지 45개 과목의 전문가 수준 한국어 객관식 평가 문항 35,030개로 구성된다. 영어 MMLU를 번역한 데이터가 아니라 한국의 실제 시험에서 수집한 문제를 사용한다.

```python
{
    "question": "맥주의 저장 시 숙성기간 동안 단백질은 무엇과 결합하여 침전하는가?",
    "A": "맥아",
    "B": "세균",
    "C": "탄닌",
    "D": "효모",
    "answer": 3,
    "Category": "Food Processing",
    "Human Accuracy": 0.1111,
}
```

`answer`의 숫자는 데이터셋의 정답 인코딩이다. 숫자가 0부터 시작하는지 1부터 시작하는지 확인하지 않고 `A`~`D`로 임의 변환하면 채점 전체가 틀어질 수 있다. 실제 evaluator가 사용하는 task 설정에서 매핑을 확인한다.

객관식 평가는 각 선택지의 로그우도를 비교하거나, 모델이 생성한 문자열에서 `A`~`D`를 추출해 채점할 수 있다.

출처: [KMMLU 원 논문](https://aclanthology.org/2025.naacl-long.206/), [KMMLU 데이터 카드](https://huggingface.co/datasets/HAERAE-HUB/KMMLU)


### KorQuAD: 문서에서 답 구간을 찾는다

KorQuAD는 한국어 기계독해(MRC)·질의응답 데이터셋이다. KorQuAD 1.0은 Wikipedia 문단에서 답 구간을 찾고, 2.0과 오류 수정 재배포본 2.1은 Wikipedia 문서 전체와 표·목록·HTML 구조까지 입력에 포함한다.

```python
context = "KorQuAD 2.0은 웹 문서 수준의 한국어 기계독해 데이터셋이다."
question = "KorQuAD 2.0은 어떤 수준의 문서를 대상으로 하는가?"
answer_text = "웹 문서 수준"

record = {
    "context": context,
    "question": question,
    "answers": {
        "text": [answer_text],
        "answer_start": [context.index(answer_text)],
    },
}
```

`answer_start`는 정답 문자열이 원문에서 시작되는 문자 위치이다. 직접 추측하지 않고 원문에서 계산해야 한다. 추출형 질의응답은 정답 문자열 전체가 같은지 보는 Exact Match와, 일부 토큰이라도 맞았는지 보는 token F1을 함께 사용한다.

출처: [KorQuAD 공식 사이트](https://korquad.github.io/), [KorQuAD 1.0 논문](https://arxiv.org/abs/1909.07005)


## 분류 지표: Accuracy, Precision, Recall, F1

양성을 `1`, 음성을 `0`이라고 할 때 먼저 예측 결과를 네 종류로 나눈다.

| 구분 | 의미 | 질문으로 바꾸면 |
|---|---|---|
| TP | 실제 양성을 양성으로 맞힌 수 | 찾아야 할 대상을 제대로 찾았는가? |
| FP | 실제 음성을 양성으로 잘못 판단한 수 | 아닌 대상을 잘못 경고했는가? |
| FN | 실제 양성을 음성으로 놓친 수 | 찾아야 할 대상을 놓쳤는가? |
| TN | 실제 음성을 음성으로 맞힌 수 | 아닌 대상을 잘 제외했는가? |

- `Accuracy = (TP + TN) / 전체`는 전체 예측 중 정답 비율이다.
- `Precision = TP / (TP + FP)`는 양성이라고 예측한 것 중 실제 양성 비율이다.
- `Recall = TP / (TP + FN)`는 실제 양성 중 찾아낸 비율이다.
- `F1 = 2 × Precision × Recall / (Precision + Recall)`은 Precision과 Recall의 조화평균이다.

Accuracy는 전체 비율을 보지만, 양성이 드문 불균형 데이터에서는 소수 클래스를 놓쳐도 높게 나올 수 있다. 아래 사례로 지표마다 사용하는 분모가 다름을 확인한다.

![이진 분류 혼동행렬](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@bbdd290451cb2e955a3e1ae4e9f76e795947aa27/08_llm/14_lm_evaluation/01_llm_eval/binary_confusion_matrix.svg)

그림의 네 칸은 위 표와 수식에서 사용한 **같은 TP·FP·FN·TN**이다. 세로축의 실제값과 가로축의 예측값이 만나는 칸을 먼저 찾고, Accuracy는 네 칸 전체, Precision은 `TP + FP`, Recall은 `TP + FN`을 분모로 사용한다.


In [ ]:
# 입력: 분류 결과를 세어 얻은 TP, FP, FN, TN이다.
tp, fp, fn, tn = 3, 1, 2, 4

# 각 지표는 관찰하려는 오류에 따라 서로 다른 분모를 사용한다.
accuracy = (tp + tn) / (tp + fp + fn + tn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
print(
    {
        "Accuracy": round(accuracy, 3),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1": round(f1, 3),
    }
)


## LLM 출력은 어떻게 채점하는가

객관식과 열린 생성은 모델에서 얻는 값과 채점 비용이 다르다.

| 구분 | 모델에서 얻는 값 | 대표 사용 | 주의점 |
|---|---|---|---|
| 로그우도 채점 | 후보별 조건부 로그확률 | BoolQ, 객관식 | 후보 길이와 길이 정규화 방식 |
| 생성 후 채점 | 모델이 생성한 문자열 | 서술형, 요약, 지시 준수 | 디코딩 설정과 정답 추출 규칙 |

로그우도 채점은 같은 prompt 뒤에 각 후보 문자열이 이어질 조건부 로그확률을 계산하고 가장 큰 후보를 고른다. 합산 로그우도는 토큰이 많을수록 더 작은 값이 되기 쉬우므로 벤치마크가 원점수와 토큰 수로 나눈 길이 정규화 점수 중 무엇을 쓰는지 확인한다.

생성 후 채점은 모델이 만든 문자열에서 정답을 추출한 뒤 Exact Match, F1 같은 지표로 기준 답과 비교한다. 정규식과 공백 제거도 점수에 영향을 주는 평가 규칙이므로 실행 설정에 기록한다.

예를 들어 모델 출력이 `C\n해설: ...`이라면 평가 규칙은 정규식 `[A-D]`로 `C`만 추출한 뒤 기준 답과 비교한다. 이 단계의 실제 모델 로그우도 계산과 생성 평가는 `02_Evaluation_LLM_lm-eval-harness.ipynb`에서 수행한다.


## 열린 생성 지표: Exact Match, BLEU, ROUGE, BERTScore

열린 생성은 같은 뜻을 여러 문장으로 표현할 수 있다. 따라서 `기준 답과 무엇이 같아야 정답인가`를 먼저 결정한다.

| 지표 | 무엇을 비교하는가 | 잘 맞는 과업 | 놓칠 수 있는 품질 |
|---|---|---|---|
| Exact Match | 정규화한 문자열 전체 일치 | 짧은 정답, 정답 선택지 | 같은 뜻의 다른 표현 |
| BLEU | 후보의 n-gram 정밀도와 짧은 문장 패널티 | 기계번역의 말뭉치 비교 | 의미 보존, 사실성, 자연스러움 |
| ROUGE | 기준 답의 n-gram 또는 공통 부분열 회수 | 요약의 핵심 내용 포함 | 사실 오류, 문장 품질 |
| BERTScore | 문맥 임베딩 기반 토큰 유사도 | 표현이 다양한 생성 답 | 외부 사실 검증, 안전성 |

`n-gram`은 연속된 n개의 token 묶음이다. 예를 들어 `대한민국의 수도`에서 unigram은 개별 token, bigram은 인접한 두 token 묶음이다. BLEU·ROUGE·BERTScore 모두 기준 답의 품질과 구현 설정에 영향을 받으므로 모델 이름, tokenizer, metric 옵션을 함께 기록한다.

앞의 수도 질문에서 `서울`과 `대한민국의 수도는 서울이다`는 의미가 같지만 Exact Match 결과는 다르다. 이처럼 열린 생성은 문자열 지표만으로 결론 내리지 않고 의미·사실성 평가를 함께 사용한다.


## 평가 자동화 도구: lm-evaluation-harness

`lm-evaluation-harness`는 모델 backend, 과업, few-shot 수, 채점 지표를 같은 실행 흐름으로 묶는 평가 프레임워크이다.

```text
모델 지정 → task 설정 로드 → 문항별 추론·채점 → 점수 집계 → sample 오류 분석
```

현재 CLI에서는 다음 순서로 과업을 찾고 설정을 검증한 뒤 평가한다.

```bash
lm-eval ls tasks
lm-eval validate --tasks <TASK_NAME>

lm-eval run \
  --model hf \
  --model_args pretrained=<MODEL_ID>,dtype=auto \
  --tasks <TASK_NAME> \
  --num_fewshot 0 \
  --device cuda:0 \
  --batch_size auto \
  --log_samples \
  --output_path results/
```

Hugging Face backend에는 `transformers`와 `accelerate`가 필요하다. 다음 실습은 `lm-eval==0.4.12` core와 HF 의존성을 각각 고정해 설치한다. `[hf]` extra를 쓰지 않는 이유는 직전 Prefix Tuning 실습의 `peft==0.18.0`을 불필요하게 업그레이드하지 않기 위해서이다.

`--log_samples`는 평균 점수뿐 아니라 개별 입력과 모델 출력을 남긴다. 평균이 변한 이유를 설명하려면 집계 결과와 실패 문항을 함께 읽어야 한다. 실제 설치와 GPU 평가는 다음 노트북에서 수행한다.

출처: [공식 CLI 문서](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/interface.md)


### task YAML은 데이터 한 행을 모델 요청으로 바꾼다

task YAML은 데이터셋의 column을 prompt, 정답, 선택지, metric에 연결한다.

```yaml
task: sample_multiple_choice
dataset_path: <DATASET_PATH>
dataset_name: <SUBSET_NAME>
output_type: multiple_choice
test_split: test
doc_to_text: "질문: {{question}}\n정답:"
doc_to_target: answer
doc_to_choice: ["{{A}}", "{{B}}", "{{C}}", "{{D}}"]
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
```

| 설정 | 역할 |
|---|---|
| `dataset_path`, `dataset_name` | 사용할 데이터와 하위 구성을 선택한다. |
| `output_type` | `multiple_choice`, `loglikelihood`, `generate_until` 등 모델 요청 방식을 정한다. |
| `doc_to_text` | 한 행을 모델 prompt로 변환한다. |
| `doc_to_target` | 정답 열이나 정답 생성 규칙을 정한다. |
| `doc_to_choice` | 객관식에서 비교할 후보를 만든다. |
| `metric_list` | 문항 점수를 전체 결과로 집계한다. |
| `generation_kwargs`, `filter_list` | 생성 길이·중단 조건과 정답 추출 규칙을 정한다. |

위 YAML은 구조를 읽기 위한 축약 예시이다. 실제 task의 정답 index와 field 이름은 데이터 schema에 맞춰 작성하고 `lm-eval validate`로 검사한다.

출처: [공식 task guide](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/task_guide.md)


## LLM-as-a-Judge

LLM-as-a-Judge는 질문, 후보 답변, 평가 기준을 심사 모델에 주고 점수·근거·선호를 받는 방식이다. 의미 보존, 문체, 복합 기준처럼 문자열 겹침만으로 보기 어려운 품질을 넓은 표본에서 비교할 수 있다.

| 사례 | 구성 | 수업에서 보는 의미 |
|---|---|---|
| MT-Bench | 80개의 2-turn 질문과 LLM 심사 | 단일 답변 채점과 쌍대 비교의 대표 사례이다. |
| LogicKor | 6개 범주, 총 42개의 한국어 multi-turn prompt | 한국어 LLM-as-a-Judge 적용 사례이며 현재 저장소는 archive 상태이다. |

심사 prompt에는 평가 축, 점수 척도, 출력 형식, 동점·불확실성 처리 규칙을 명시한다. 심사 모델에는 다음 편향이 생길 수 있다.

- 자기 선호는 심사 모델이 자신 또는 같은 계열 모델의 답을 선호하는 현상이다.
- 위치 편향은 두 답 중 먼저 또는 나중에 제시된 답을 선호하는 현상이다.
- 장문 편향은 내용보다 답변 길이에 높은 점수를 주는 현상이다.

사람 평가 표본과 일치도를 확인하고, 쌍대 비교에서는 모델 이름을 가린 뒤 제시 순서를 바꿔 편향을 점검한다. Pearson 상관계수는 두 점수의 선형적인 동행 정도를 보여 줄 뿐, 심사 점수의 정확성·인과관계·개별 답변의 품질을 보증하지 않는다.

출처: [MT-Bench 원 논문](https://proceedings.neurips.cc/paper_files/paper/2023/hash/91f18a1287b398d378ef22505bf41832-Abstract-Datasets_and_Benchmarks.html), [LogicKor 저장소](https://github.com/instructkr/LogicKor)

실제 심사 모델 호출, 점수 parsing과 사람 평가의 상관관계 비교는 `03_llm_as_a_judge_hf_cookbook.ipynb`에서 수행한다.


## RAG 평가는 단계를 나누어 본다

RAG 답변의 실패는 검색 문서가 없어서 생길 수도 있고, 검색 문서는 맞지만 모델이 근거를 사용하지 않아 생길 수도 있다. 질문, 검색 문맥, 생성 답변, 필요하면 기준 답을 분리해 평가한다.

| 관찰 대상 | 확인하는 질문 | 다음 개선 후보 |
|---|---|---|
| 검색 문맥 | 질문에 필요한 문서가 검색되었는가? | chunk, embedding, 검색기, top-k |
| 근거 충실성 | 답변이 검색 문맥으로 뒷받침되는가? | prompt, citation, 생성 제약 |
| 답변 관련성 | 질문에 직접 답했는가? | prompt, model, 후처리 |
| 정답 정확성 | 기준 답과 핵심 사실이 일치하는가? | 검색과 생성 단계를 함께 점검한다. |

RAGAS 같은 프레임워크는 이러한 관점을 나누어 측정하는 데 사용한다. 프레임워크 점수만으로 결론 내리지 않고 실제 근거 인용과 실패 사례를 사람이 확인한다. RAGAS의 0~1 점수도 해당 데이터셋·심사 모델·설정 안에서 비교하는 상대적 신호이며, 서비스 품질의 절대 백분율로 해석하지 않는다. 이 노트북에서는 위치만 구분하고 실제 RAGAS 코드는 RAG 평가 노트북에서 다룬다.


## 평가 결과로 다음 실험 정하기

좋은 평가는 순위표를 만드는 데서 끝나지 않는다. 점수와 sample을 함께 읽어 실패 원인을 다음 실험으로 연결한다.

| 관찰 결과 | 가능한 원인 | 다음 실험 |
|---|---|---|
| 객관식 지식 점수는 낮지만 형식 오류는 적다. | 모델 지식 또는 추론 부족 | 모델·few-shot·CoT 조건을 비교한다. |
| 생성 답의 내용은 맞지만 Exact Match가 낮다. | 출력 형식 또는 추출 규칙 문제 | 정규화·regex와 prompt 형식을 비교한다. |
| RAG 답에 근거 없는 내용이 많다. | 검색 누락 또는 문맥 미사용 | 검색 metric과 충실성을 분리한다. |
| Judge 점수만 높고 사람 평가가 낮다. | 심사 rubric 또는 편향 문제 | 사람 표본, 순서 교환, blind 평가를 추가한다. |

비교 실험에서는 모델 외의 조건을 고정하고 데이터셋 버전, 명령, 모델 ID, seed, few-shot 수, metric, 후처리를 기록한다. 이 기록이 있어야 다음 실행의 점수 차이를 재현 가능한 변화로 해석할 수 있다.


## 부록: 모델 카드에 등장하는 벤치마크 범주

본문에서는 선택 기준을 빠르게 읽도록 대표 항목만 비교했다. 아래 목록은 모델 카드의 범주와 주요 벤치마크를 한 번에 확인하기 위한 것이다.

### Base - General

- MMLU, MMLU-Pro(CoT), AGIEval English
- CommonSenseQA, Winogrande
- BIG-Bench Hard(CoT), ARC-Challenge

### Base - Knowledge·Reading

- TriviaQA-Wiki, SQuAD, QuAC
- BoolQ, DROP

### Instruction-tuned - 지시·추론·코드·수학·도구

- IFEval, ARC-C, GPQA
- HumanEval, MBPP·MBPP++
- GSM8K(CoT), MATH(CoT)
- API-Bank, BFCL, Gorilla API Bench, Nexus(0-shot)

### Multilingual

- Multilingual MMLU
- 다국어 ARC, HellaSwag, MMLU 번역·확장 세트

각 벤치마크는 같은 이름이라도 version, split, shots, metric이 달라질 수 있다. 실제 비교에서는 사용한 task 설정을 결과와 함께 보관한다.


## 결론

LLM 평가는 먼저 과업의 성공 조건을 정의하고, 그 조건을 관찰할 수 있는 데이터와 지표를 선택하는 일이다. 분류와 객관식은 Accuracy·F1·로그우도를 사용하기 쉽고, 번역·요약은 BLEU·ROUGE·BERTScore를 보조적으로 사용할 수 있다. 복합적인 답변 품질은 사람 평가와 편향을 점검한 LLM 심사를 함께 사용한다.

벤치마크 평균 하나를 최종 결론으로 사용하지 않는다. 표준 벤치마크, 실제 서비스 평가셋, 자동 지표, 사람 검토, 오류 사례를 조합하고 실행 조건을 기록해야 재현 가능하고 개선으로 이어지는 평가가 된다.

### 참고 자료

- [BLEU 원 논문](https://aclanthology.org/P02-1040/)
- [ROUGE 원 논문](https://aclanthology.org/W04-1013/)
- [BERTScore 원 논문](https://openreview.net/forum?id=SkeHuCVFDr)
- [lm-evaluation-harness 공식 저장소](https://github.com/EleutherAI/lm-evaluation-harness)
